In [ ]:
# ============================================================
# CIFAR-10 IMAGE CLASSIFICATION PROJECT - HIGH ACCURACY CNN
# ============================================================

#Bu kısım, projede kullanılacak kütüphaneleri (hazır araçları) programa ekler. Kısaca görevleri:

#numpy (np) → Sayısal işlemler, dizi/matris hesaplamaları yapar.
#matplotlib.pyplot (plt) → Grafik çizmek için kullanılır.
#tensorflow (tf) → Derin öğrenme modeli oluşturmak ve eğitmek için ana kütüphane.
#random → Rastgele seçimler yapar.
#os → Dosya ve klasör işlemleri yapar.
#Keras Modülleri:
#cifar10 → CIFAR-10 veri setini hazır yükler.
#to_categorical → Etiketleri one-hot formatına çevirir.
#Sequential → Katman katman CNN modeli kurar.
#Conv2D → Görsellerden özellik çıkaran konvolüsyon katmanı.
#MaxPooling2D → Görsel boyutunu küçültür, önemli bilgiyi korur.
#BatchNormalization → Eğitimi dengeler ve hızlandırır.
#Dropout → Aşırı ezberlemeyi önler.
#Flatten → Veriyi düz hale getirir.
#Dense → Tam bağlantılı sinir ağı katmanı.
#Callbacks:
#EarlyStopping → Model kötüleşirse eğitimi durdurur.
#ReduceLROnPlateau → Öğrenme hızını düşürür.
#ModelCheckpoint → En iyi modeli kaydeder.
#Veri Artırma:
#ImageDataGenerator → Görselleri döndürme, kaydırma vb. ile çoğaltır.
#Performans Ölçümü:
#classification_report → Accuracy, precision, recall verir.
#confusion_matrix → Doğru/yanlış tahmin tablosu çıkarır.
#Görselleştirme:
#seaborn (sns) → Güzel grafikler ve confusion matrix çizimi yapar.

# Gerekli kütüphaneler
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import random
import os

from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, BatchNormalization, Dropout,
    Flatten, Dense
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# ============================================================
# 1. SABİT TOHUMLAR
# ============================================================

#Bu bölüm, programın her çalıştırıldığında aynı sonuçları vermesi için kullanılır. Buna rastgelelik kontrolü (seed ayarı) denir.

#Satırların görevi:
#SEED = 42
#Ortak sabit sayı belirlenir. (Genelde 42 sembolik olarak sık kullanılır.)
#np.random.seed(SEED)
#NumPy içindeki rastgele işlemleri sabitler.
#tf.random.set_seed(SEED)
#TensorFlow içindeki rastgele işlemleri sabitler.
#random.seed(SEED)
#Python’un kendi random modülünü sabitler.
#Neden gerekli?

#CNN modeli eğitirken:

#ağırlıklar rastgele başlar
#veri karıştırılır
#dropout rastgele çalışır

#Bu ayarlar yapılırsa sonuçlar daha tutarlı olur.

#Kısaca:

#Bugün çalıştırdığın model ile yarın çalıştırdığın model benzer sonuç verir.
#SEED = 42
#np.random.seed(SEED)
#tf.random.set_seed(SEED)
#random.seed(SEED)

# ============================================================
# 2. DATASET YÜKLEME
# ============================================================

#Bu bölüm, CIFAR-10 veri setini yükler ve temel bilgileri gösterir.
#Satırların görevi:

#print("CIFAR-10 dataseti yükleniyor...")
#Ekrana bilgi mesajı verir.

#(x_train, y_train), (x_test, y_test) = cifar10.load_data()
#Veri setini indirir/yükler.

#Burada:
#x_train → Eğitim resimleri
#y_train → Eğitim etiketleri
#x_test → Test resimleri
#y_test → Test etiketleri

#CIFAR-10 İçeriği:
#Toplam 60.000 adet 32x32 renkli resim vardır.
#50.000 eğitim
#10.000 test

#class_names:
#Bu liste sınıf isimlerini tutar:
#airplane
#automobile
#bird
#cat
#deer
#dog
#frog
#horse
#ship
#truck

#Model tahmin yaptığında sayı yerine isim göstermek için kullanılır.
#Son iki print:

#x_train.shape → Eğitim veri boyutu
#Genelde: (50000, 32, 32, 3)

#x_test.shape → Test veri boyutu
#Genelde: (10000, 32, 32, 3)

#Bu sayıların anlamı:
#50000 resim
#32x32 piksel
#3 kanal (RGB renk)

#Kısaca:
#Bu kod, resmi ve etiketleri yükler, sınıf isimlerini tanımlar ve veri boyutunu kontrol eder.

print("CIFAR-10 dataseti yükleniyor...")

(x_train, y_train), (x_test, y_test) = cifar10.load_data()

class_names = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]

print("Eğitim veri boyutu:", x_train.shape)
print("Test veri boyutu:", x_test.shape)

# ============================================================
# 3. NORMALİZASYON
# ============================================================

#Bu bölüm, veriyi modelin daha iyi öğrenebilmesi için hazırlar.

#1. Normalizasyon
#x_train = x_train.astype("float32") / 255.0
#x_test = x_test.astype("float32") / 255.0
#Ne yapıyor?
#Resim pikselleri normalde:

#0 = siyah
#255 = beyaz
#aradaki değerler gri / renk tonlarıdır.
#Bu kod tüm piksel değerlerini 0 ile 1 arasına çevirir.

#Örnek:

#0 → 0.00
#128 → 0.50
#255 → 1.00
#Neden gerekli?

#Çünkü TensorFlow / CNN modeli küçük sayılarla daha hızlı ve daha stabil öğrenir.

#2. astype("float32")

#Resim verisini tam sayıdan (integer) ondalıklı sayıya çevirir.

#Çünkü:

#255 / 255 = 1

#gibi işlemler float olarak daha doğru yapılır.

#3. One-Hot Encoding
#y_train_cat = to_categorical(y_train, 10)
#y_test_cat = to_categorical(y_test, 10)

#Etiketleri sayıdan vektöre çevirir.

#Örneğin:

#cat = 3

#Normal hali:

#3

#One-hot hali:

#[0,0,0,1,0,0,0,0,0,0]

#10 sınıf olduğu için uzunluk 10’dur.

#Neden gerekli?
#Çünkü çıkış katmanında 10 sınıf vardır ve model her sınıf için olasılık üretir.
#Örneğin:

#[0.01,0.02,0.03,0.85,0.01,...]

#En yüksek değer = tahmin edilen sınıf.

#Kısaca:

#Bu bölüm:

#Resimleri 0-1 aralığına çevirir
#Etiketleri modele uygun hale getirir
#Eğitime hazır veri oluşturur

x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# One-hot encoding
y_train_cat = to_categorical(y_train, 10)
y_test_cat = to_categorical(y_test, 10)

# ============================================================
# 4. DATA AUGMENTATION
# ============================================================

#Bu bölüm, veri artırma (Data Augmentation) yapar. Amaç, mevcut resimleri farklı şekillerde değiştirerek modele daha fazla örnek göstermektir.
#Neden yapılır?

#Gerçek hayatta nesneler hep aynı açıdan gelmez:

#biraz eğik olabilir
#sağda solda olabilir
#yakın/uzak olabilir
#ters yönde olabilir

#Bu yöntem modelin ezberlemesini azaltır, genellemesini artırır.

#ANA KOD:
#datagen = ImageDataGenerator(...)

#TensorFlow içindeki görüntü artırma aracıdır.

#PARAMETRELER
#rotation_range=15
#Resmi rastgele ±15 derece döndürür.

#width_shift_range=0.10
#Resmi yatayda %10 sağa/sola kaydırır.

#height_shift_range=0.10
#Resmi dikeyde %10 yukarı/aşağı kaydırır.

#horizontal_flip=True
#Resmi sağ-sol aynalar.

#Örnek:

#araba sola bakıyor → sağa bakıyor
#zoom_range=0.10
#Resmi %10 yakınlaştırır/uzaklaştırır.

#validation_split=0.1
#Eğitim verisinin %10’unu doğrulama verisi yapar.

#Yani:

#%90 train
#%10 validation


#TRAIN GENERATOR
#subset="training"

#Bu kısım eğitim verisini üretir.

#batch_size=64

#Modele her adımda 64 resim verilir.

#shuffle=True

#Veriler karıştırılır. Öğrenme için iyidir.

#VAL_GENERATOR
#subset="validation"

#Bu kısım doğrulama verisidir.

#shuffle=False

#Sıralı kalır. Ölçüm için uygundur.

#Kısaca:

#Bu bölüm:

#Resimleri çoğaltır
#Döndürür / kaydırır / aynalar / zoom yapar
#Train ve validation olarak ayırır
#Modelin daha yüksek doğruluk vermesine yardım eder

datagen = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.10,
    height_shift_range=0.10,
    horizontal_flip=True,
    zoom_range=0.10,
    validation_split=0.1
)

train_generator = datagen.flow(
    x_train, y_train_cat,
    batch_size=64,
    subset="training",
    shuffle=True
)

val_generator = datagen.flow(
    x_train, y_train_cat,
    batch_size=64,
    subset="validation",
    shuffle=False
)

# ============================================================
# 5. GELİŞMİŞ CNN MODELİ
# ============================================================
model = Sequential()

# Blok 1
model.add(Conv2D(32, (3, 3), padding="same", activation="relu", input_shape=(32, 32, 3)))
model.add(BatchNormalization())
model.add(Conv2D(32, (3, 3), padding="same", activation="relu"))
model.add(BatchNormalization())
model.add(MaxPooling2D((2, 2)))
model.add(Dropout(0.25))

# Blok 2
model.add(Conv2D(64, (3, 3), padding="same", activation="relu"))
model.add(BatchNormalization())
model.add(Conv2D(64, (3, 3), padding="same", activation="relu"))
model.add(BatchNormalization())
model.add(MaxPooling2D((2, 2)))
model.add(Dropout(0.30))

# Blok 3
model.add(Conv2D(128, (3, 3), padding="same", activation="relu"))
model.add(BatchNormalization())
model.add(Conv2D(128, (3, 3), padding="same", activation="relu"))
model.add(BatchNormalization())
model.add(MaxPooling2D((2, 2)))
model.add(Dropout(0.35))

# Sınıflandırma katmanları
model.add(Flatten())
model.add(Dense(256, activation="relu"))
model.add(BatchNormalization())
model.add(Dropout(0.5))
model.add(Dense(10, activation="softmax"))

model.summary()

# ============================================================
# 6. MODEL DERLEME
# ============================================================

#MODEL DERLEME
#Bu bölüm, modeli eğitime hazır hale getirir.
#model.compile(...)

#OPTIMIZER
#Adam(learning_rate=0.001)
#Modelin hataları azaltarak öğrenmesini sağlar.
#Adam, en sık kullanılan ve güçlü optimizasyon yöntemlerinden biridir.


#learning_rate=0.001 → Öğrenme hızı

#LOSS
#categorical_crossentropy
#Modelin tahminleri ile gerçek cevap arasındaki hatayı hesaplar.
#Bu loss, çok sınıflı sınıflandırma için uygundur.
#(CIFAR-10 = 10 sınıf)

#METRICS
#accuracy
#Modelin yüzde kaç doğru tahmin yaptığını gösterir.
#Örnek:

#%85 accuracy = 100 resmin 85’i doğru

#KISACA
#Bu bölümde:
#Nasıl öğreneceği seçilir
#Hata hesabı belirlenir
#Başarı ölçütü tanımlanır
#Model eğitime hazır olur

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
    )

# ============================================================
# 7. CALLBACKS
# ============================================================

#Bu bölüm, eğitim sırasında modeli otomatik kontrol eden yardımcı sistemlerdir.

#CHECKPOINT PATH
#checkpoint_path = "best_cifar10_model.keras"
#En iyi model bu dosyaya kaydedilir.

#EARLYSTOPPING
#EarlyStopping(...)
#Model gelişmiyorsa eğitimi durdurur.


#monitor="val_accuracy" → Doğrulama başarısını izler
#patience=8 → 8 epoch gelişmezse durur
#restore_best_weights=True → En iyi ağırlıkları geri yükler
#Amaç: Gereksiz eğitimi önlemek.

#REDUCELRONPLATEAU
#ReduceLROnPlateau(...)
#Model takılırsa öğrenme hızını düşürür.


#monitor="val_loss" → Hata değerini izler
#factor=0.5 → Learning rate yarıya iner
#patience=4 → 4 epoch gelişme yoksa düşürür
#min_lr=1e-6 → En düşük sınır
#Amaç: Daha hassas öğrenme sağlamak.

#MODELCHECKPOINT
#ModelCheckpoint(...)
#En iyi modeli otomatik kaydeder.


#save_best_only=True → Sadece en iyi model kaydedilir
#monitor="val_accuracy" → Başarıya göre seçer

#KISACA
#Bu bölüm:
#Gereksiz eğitimi durdurur
#Öğrenme hızını ayarlar
#En iyi modeli kaydeder
#Daha yüksek performans sağlar

checkpoint_path = "best_cifar10_model.keras"

callbacks = [
    EarlyStopping(
        monitor="val_accuracy",
        patience=8,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=4,
        min_lr=1e-6,
        verbose=1
    ),
    ModelCheckpoint(
        checkpoint_path,
        monitor="val_accuracy",
        save_best_only=True,
        verbose=1
    )
]

# ============================================================
# 8. MODEL EĞİTİMİ
# ============================================================

#Bu bölümde model, verileri kullanarak öğrenmeye başlar.
#model.fit(...)

#TRAIN_GENERATOR
#train_generator
#Eğitim verilerini modele verir.
#(Artırılmış resimler kullanılır.)

#VALIDATION_DATA
#val_generator
#Her epoch sonunda model test edilir.
#Ne kadar iyi öğrendiği kontrol edilir.

#EPOCHS
#epochs=40
#Tüm eğitim verisi en fazla 40 kez modele gösterilir.

#Eğer callback erken durdurursa 40’a ulaşmadan bitebilir.


#CALLBACKS
#callbacks=callbacks
#Önceki bölümde tanımlanan:

#EarlyStopping
#ReduceLROnPlateau
#ModelCheckpoint aktif olur.

#VERBOSE
#verbose=1
#Eğitim sırasında ekranda ilerleme bilgisi gösterilir.
#Örnek:
#Epoch 1/40loss: ...accuracy: ...val_accuracy: ...

#HISTORY
#history =
#Eğitim sonuçlarını kaydeder.
#İçinde:

#loss
#accuracy
#val_loss
#val_accuracy bulunur.
#Sonradan grafik çizmekte kullanılır.

#KISACA
#Bu bölüm:

#Modeli eğitir
#Her tur performansı ölçer
#En iyi sonucu takip eder
#Sonuçları history içine kaydeder

history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=40,
    callbacks=callbacks,
    verbose=1
)

# ============================================================
# 9. EĞİTİM GRAFİKLERİ
# ============================================================

#Bu bölüm, model eğitim sonuçlarını grafik olarak gösterir.

#FIGURE OLUŞTURMA
#plt.figure(figsize=(12,5))
#12x5 boyutunda grafik alanı oluşturur.

#SOL GRAFİK: ACCURACY
#plt.subplot(1,2,1)
#1 satır 2 sütunluk alanın ilk grafiği.

#Train Accuracy
#Validation Accuracy

#Gösterir:

#Eğitim başarısı
#Doğrulama başarısı

#X ekseni = Epoch
#Y ekseni = Accuracy

#SAĞ GRAFİK: LOSS
#plt.subplot(1,2,2)

#İkinci grafik alanı.

#Gösterir:

#Train Loss
#Validation Loss

#X ekseni = Epoch
#Y ekseni = Loss

#TIGHT LAYOUT
#plt.tight_layout()
#Grafiklerin düzgün hizalanmasını sağlar.

#SHOW
#plt.show()
#Grafiği ekranda gösterir.

#Bu bölüm sayesinde:

#Model gelişiyor mu görülür
#Overfitting var mı anlaşılır
#Accuracy artışı izlenir
#Loss düşüşü izlenir

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history.history["accuracy"], label="Train Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.title("Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history["loss"], label="Train Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.title("Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.tight_layout()
plt.show()

# ============================================================
# 10. TEST DEĞERLENDİRME
# ============================================================

#Bu bölüm, eğitilen modelin daha önce görmediği test verisindeki gerçek performansını ölçer.

#MODEL EVALUATE
#model.evaluate(x_test, y_test_cat, verbose=0)
#Model, test resimlerini tahmin eder ve sonuçları hesaplar.

#x_test → Test resimleri
#y_test_cat → Gerçek cevaplar

#ÇIKAN SONUÇLAR
#test_loss
#Modelin hata oranı.
#Düşük olması iyidir.

#test_acc
#Modelin doğruluk oranı.
#Yüksek olması iyidir.

#VERBOSE=0
#verbose=0
#Ekranda işlem detaylarını göstermez.

#PRINT SATIRLARI
#round(...,4)
#Sonuçları virgülden sonra 4 basamak gösterir.
#Örnek:
#Test Loss: 0.4217Test Accuracy: 0.8735
#Yani:
#Hata = 0.4217
#Doğruluk = %87.35

#KISACA
#Bu bölüm:
#Modelin gerçek başarısını ölçer
#Daha önce görmediği veride test eder
#Accuracy ve Loss verir
#Projenin final sonucunu gösterir

test_loss, test_acc = model.evaluate(x_test, y_test_cat, verbose=0)

print("\nTest Loss:", round(test_loss, 4))
print("Test Accuracy:", round(test_acc, 4))

# ============================================================
# 11. TAHMİNLER
# ============================================================

#Bu bölüm, modelin test resimleri için hangi sınıfı tahmin ettiğini bulur.

#MODEL PREDICT
#y_pred_probs = model.predict(x_test, verbose=0)

#Model test resimlerini tahmin eder.

#Sonuç olarak her resim için 10 sınıfa ait olasılık verir.

#Örnek:

#[0.01, 0.02, 0.80, 0.03, ...]

#Yani:
#airplane %1
#automobile %2
#bird %80
#...

#ARGMAX
#y_pred = np.argmax(y_pred_probs, axis=1)

#En büyük olasılığın indeksini alır.

#Örnek:

[0.01,0.02,0.80,...]

#En büyük değer = index 2

#Yani tahmin edilen sınıf = bird

#GERÇEK ETİKETLER
#y_true = y_test.flatten()

#Gerçek cevapları tek boyutlu hale getirir.

#Önce:

#[[3],[5],[1]]

#Sonra:
#[3,5,1]

#VERBOSE=0
#Ekranda tahmin sürecini göstermez.

#KISACA

#Bu bölüm:

#Test resimlerini tahmin eder
#En yüksek olasılığı seçer
#Gerçek etiketleri hazırlar
#Sonraki başarı analizleri için veri oluşturur

y_pred_probs = model.predict(x_test, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = y_test.flatten()

# ============================================================
# 12. CLASSIFICATION REPORT
# ============================================================

#Bu bölüm, modelin her sınıf için ne kadar başarılı olduğunu detaylı gösterir.

#ANA KOD
#classification_report(y_true, y_pred, target_names=class_names)

#Karşılaştırır:
#y_true → Gerçek cevaplar
#y_pred → Model tahminleri

#TARGET_NAMES
#class_names
#Sayı yerine sınıf isimlerini yazar:

#airplane
#cat
#dog
#ship
#truck ...

#RAPORDA ÇIKAN DEĞERLER#
#PRECISION
#Model o sınıf dediğinde ne kadar doğru?
#RECALL
#Gerçekte o sınıf olanları ne kadar buldu?
#F1-SCORE
#Precision ve Recall dengeli ortalaması.
#SUPPORT
#O sınıftan kaç test resmi var.

#ÖRNEK
#cat   precision 0.82   recall 0.79

#Yani:
#Kedi dediğinde %82 doğru
#Gerçek kedilerin %79’unu yakaladı

#GENEL SONUÇ

#Altta ayrıca:

#accuracy
#macro avg
#weighted avg değerleri çıkar.

#KISACA

#Bu bölüm:

#Her sınıfın başarısını ayrı gösterir
#Hangi sınıfta zayıf olduğunu buldurur
#Accuracy’den daha detaylı analiz sağlar
#Model performans raporudur


print("\nCLASSIFICATION REPORT:\n")
print(classification_report(y_true, y_pred, target_names=class_names))

# ============================================================
# 13. CONFUSION MATRIX
# ============================================================

#Bu bölüm, modelin hangi sınıfları birbiriyle karıştırdığını gösterir.

#CONFUSION MATRIX OLUŞTURMA
#cm = confusion_matrix(y_true, y_pred)
#Karşılaştırır:

#y_true → Gerçek sınıf
#y_pred → Tahmin edilen sınıf

#Sonuç bir tablo olur.

#GRAFİK BOYUTU
#plt.figure(figsize=(10,8))
#10x8 boyutunda grafik alanı açar.

#HEATMAP ÇİZME
#sns.heatmap(...)
#Seaborn ile renkli tablo çizer.

#PARAMETRELER
#cm → confusion matrix verisi
#annot=False → sayı yazmaz
#cmap="Blues" → mavi tonları kullanır
#xticklabels → Tahmin edilen sınıflar
#yticklabels → Gerçek sınıflar

#EKSENLER
#Predicted
#X ekseni = Model tahmini
#Y ekseni = Gerçek sınıf

#NASIL YORUMLANIR?
#Köşegen (çapraz çizgi) Doğru tahminlerdir.Ne kadar koyuysa o kadar iyi.Köşegen dışı alanlarKarıştırılan sınıflardır.
#Örnek:
#cat → dog
#truck → automobile

#KISACA
#Bu bölüm:
#Hangi sınıflar doğru tahmin edildi gösterir
#Hangi sınıflar karıştı gösterir
#Model zayıflıklarını buldurur
#Görsel performans analizidir


cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=False, cmap="Blues", xticklabels=class_names, yticklabels=class_names)
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()

# ============================================================
# 14. DOĞRU VE YANLIŞ TAHMİN ÖRNEKLERİ
# ============================================================

#Bu bölüm, modelin tahmin ettiği resimleri görsel olarak gösterir.

#FONKSİYON TANIMI
#def show_sample_predictions(...)
#Rastgele seçilen resimleri ekrana basar.

#RANDOM RESİM SEÇME
#indices = np.random.choice(len(images), n, replace=False)
#Test verisinden rastgele n adet resim seçer.
#Burada:
#n=15
#Yani 15 resim gösterilecek.

#GRAFİK ALANI
#plt.figure(figsize=(18,10))
#Büyük bir pencere oluşturur.

#DÖNGÜ
#for i, idx in enumerate(indices):
#Seçilen her resmi tek tek gösterir.

#RESMİ GÖSTERME
#plt.imshow(images[idx])
#Resmi ekrana basar.

#GERÇEK VE TAHMİN İSİMLERİ
#true_namepred_name
#Sayı yerine sınıf adı yazar:
#cat
#dog
#ship vb.

#RENK KONTROLÜ
#green Doğru tahmin
#red Yanlış tahmin

#BAŞLIK
#Gerçek: cat  Tahmin: dog
#şeklinde gösterir.

#AXIS OFF
#plt.axis("off")
#Kenar çizgilerini kaldırır.

#FONKSİYONU ÇALIŞTIRMA
#show_sample_predictions(..., n=15)
#15 örnek sonucu gösterir.

#KISACA
#Bu bölüm:
#Modelin gerçek tahminlerini gösterir
#Doğru / yanlış tahminleri renkle ayırır
#Hangi resimlerde zorlandığını gösterir


def show_sample_predictions(images, true_labels, pred_labels, class_names, n=15):
    indices = np.random.choice(len(images), n, replace=False)
    plt.figure(figsize=(18, 10))
    for i, idx in enumerate(indices):
        plt.subplot(3, 5, i + 1)
        plt.imshow(images[idx])
        true_name = class_names[true_labels[idx]]
        pred_name = class_names[pred_labels[idx]]
        color = "green" if true_labels[idx] == pred_labels[idx] else "red"
        plt.title(f"Gerçek: {true_name}\nTahmin: {pred_name}", color=color, fontsize=10)
        plt.axis("off")
    plt.tight_layout()
    plt.show()

print("\nÖrnek tahminler:")
show_sample_predictions(x_test, y_true, y_pred, class_names, n=15)

# ============================================================
# 15. EN YÜKSEK GÜVENLİ 10 DOĞRU TAHMİN
# ============================================================

#Bu bölüm, modelin en emin olduğu ve doğru bildiği 10 resmi gösterir.

#DOĞRU TAHMİNLERİ BULMA
#correct_indices = np.where(y_true == y_pred)[0]
#Gerçek etiket ile tahmin aynıysa seçilir.
#Yani sadece doğru tahminler alınır.

#GÜVEN SKORLARI
#correct_confidences = y_pred_probs[correct_indices, y_pred[correct_indices]]
#Her doğru tahmin için modelin verdiği olasılık alınır.
#Örnek:
#cat = 0.99
#ship = 0.97

#Bu, modelin ne kadar emin olduğunu gösterir.

#EN YÜKSEK 10 TANESİ
#top_correct = ...
#Güven skorlarına göre sıralar ve en yüksek 10 taneyi seçer.

#GRAFİK ALANI
#plt.figure(figsize=(16,8))
#10 resmi gösterecek alan oluşturur.

#RESMİ GÖSTERME
#plt.imshow(x_test[idx])
#Seçilen test resmini gösterir.

#BAŞLIK
#cat0.99
#Üstte:
#Sınıf adı
#Güven oranı yazılır.

#YEŞİL RENK
#color="green"
#Doğru tahmin olduğu için yeşil.

#KISACA
#Bu bölüm:
#Modelin en emin olduğu doğru sonuçları gösterir
#Güçlü tahminleri analiz eder
#Modelin hangi resimlerde çok başarılı olduğunu gösterir


correct_indices = np.where(y_true == y_pred)[0]
correct_confidences = y_pred_probs[correct_indices, y_pred[correct_indices]]
top_correct = correct_indices[np.argsort(correct_confidences)[-10:]]

plt.figure(figsize=(16, 8))
for i, idx in enumerate(top_correct):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_test[idx])
    plt.title(
        f"{class_names[y_true[idx]]}\n{y_pred_probs[idx][y_pred[idx]]:.2f}",
        fontsize=10,
        color="green"
    )
    plt.axis("off")
plt.tight_layout()
plt.show()

# ============================================================
# 16. EN YÜKSEK GÜVENLİ 10 YANLIŞ TAHMİN
# ============================================================

#Bu bölüm, modelin çok emin olduğu halde yanlış bildiği 10 resmi gösterir.
#Bu çok önemli analizdir çünkü modelin nerede hata yaptığını gösterir.

#YANLIŞ TAHMİNLERİ BULMA
#wrong_indices = np.where(y_true != y_pred)[0]
#Gerçek etiket ile tahmin farklıysa seçilir.
#Yani sadece yanlış tahminler alınır.

#IF KONTROLÜ
#if len(wrong_indices) > 0:
#Yanlış tahmin varsa devam eder.

#GÜVEN SKORLARI
#wrong_confidences = ...
#Yanlış tahmin ettiği halde modelin verdiği güven oranı alınır.
#Örnek:

#dog dedi %0.98
#ama gerçek = cat

#EN YÜKSEK 10 YANLIŞ
#top_wrong = ...
#En yüksek güvenli yanlış 10 örnek seçilir.

#GRAFİK GÖSTERİMİ
#plt.imshow(x_test[idx])
#Yanlış tahmin edilen resmi gösterir.

#BAŞLIK
#G: cat
#T: dog
#0.97

#G = Gerçek sınıf
#T = Tahmin edilen sınıf
#Alt sayı = güven oranı


#KIRMIZI RENK
#color="red"
#Yanlış tahmin olduğu için kırmızı.

#NEDEN ÖNEMLİ?
#Bu bölüm sayesinde:

#cat ile dog karışıyor mu?
#truck ile automobile karışıyor mu?
#düşük kaliteli resim mi?
#model aşırı özgüvenli mi? anlaşılır.

#KISACA
#Bu bölüm:
#Modelin kritik hatalarını gösterir
#Çok emin olup yanlış yaptığı örnekleri bulur
#Geliştirme fırsatlarını ortaya çıkarır
#Profesyonel analiz kısmıdır

wrong_indices = np.where(y_true != y_pred)[0]

if len(wrong_indices) > 0:
    wrong_confidences = y_pred_probs[wrong_indices, y_pred[wrong_indices]]
    top_wrong = wrong_indices[np.argsort(wrong_confidences)[-10:]]

    plt.figure(figsize=(16, 8))
    for i, idx in enumerate(top_wrong):
        plt.subplot(2, 5, i + 1)
        plt.imshow(x_test[idx])
        plt.title(
            f"G: {class_names[y_true[idx]]}\nT: {class_names[y_pred[idx]]}\n{y_pred_probs[idx][y_pred[idx]]:.2f}",
            fontsize=10,
            color="red"
        )
        plt.axis("off")
    plt.tight_layout()
    plt.show()

# ============================================================
# 17. TEK GÖRSEL İÇİN TAHMİN FONKSİYONU
# ============================================================

#Bu bölüm, modele tek bir resmi verip tahmin alma işlemini yapar.

#FONKSİYON TANIMI
#def predict_single_image(image_array):
#Tek resim alır ve sonucu döndürür.

#GÖRSEL BOYUTU
#(32,32,3)
#Bu:
#32 piksel genişlik
#32 piksel yükseklik
#3 renk kanalı (RGB) demektir.

#FLOAT'A ÇEVİRME
#astype("float32")
#Sayısal işlemler için uygun hale getirir.

#NORMALİZASYON
#if img.max() > 1.0:    img = img / 255.0
#Eğer resim 0-255 aralığındaysa, 0-1 aralığına çevirir.

#BOYUT EKLEME
#np.expand_dims(img, axis=0)
#Tek resmi modele uygun hale getirir.
#Önce:
#(32,32,3)
#Sonra:
#(1,32,32,3)
#Yani 1 adet resim.

#TAHMİN
#model.predict(img)
#10 sınıf için olasılık üretir.

#EN YÜKSEK SINIF
#np.argmax(pred)
#En büyük olasılığın sınıfını seçer.

#RETURN SONUCU
#Fonksiyon şunu döndürür:
#{ predicted_class, confidence, all_probabilities}
#predicted_class
#Tahmin edilen sınıf adı
#confidence
#En yüksek güven oranı
#all_probabilities
#Tüm sınıfların olasılığı

#ÖRNEK KULLANIM
#sample_index = 25
#test resmi seçilir.

#SONUÇ YAZDIRMA
#Gerçek sınıfTahminGüven
#Gösterilir.

#GÖRSELİ GÖSTERME
#plt.imshow(...)
#Seçilen resmi ekrana basar.

#KISACA
#Bu bölüm:
#Tek resimle tahmin yapar
#Güven oranı verir
#Gerçek ve tahmini karşılaştırır
#Gerçek projelerde kullanım için çok önemlidir


def predict_single_image(image_array):
    """
    image_array şekli: (32, 32, 3)
    """
    img = image_array.astype("float32")
    if img.max() > 1.0:
        img = img / 255.0

    img = np.expand_dims(img, axis=0)
    pred = model.predict(img, verbose=0)[0]
    class_index = np.argmax(pred)

    return {
        "predicted_class": class_names[class_index],
        "confidence": float(np.max(pred)),
        "all_probabilities": {class_names[i]: float(pred[i]) for i in range(len(class_names))}
    }

# Örnek kullanım
sample_index = 25
result = predict_single_image(x_test[sample_index])

print("\nTek görsel tahmin sonucu:")
print("Gerçek sınıf :", class_names[y_true[sample_index]])
print("Tahmin       :", result["predicted_class"])
print("Güven        :", round(result["confidence"], 4))

plt.figure(figsize=(4, 4))
plt.imshow(x_test[sample_index])
plt.title(f"Gerçek: {class_names[y_true[sample_index]]}\nTahmin: {result['predicted_class']}")
plt.axis("off")
plt.show()

print("\nModel eğitimi ve değerlendirmesi tamamlandı.")

CIFAR-10 dataseti yükleniyor...
170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
Eğitim veri boyutu: (50000, 32, 32, 3)
Test veri boyutu: (10000, 32, 32, 3)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 32, 32, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 32, 32, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 32, 32, 32)     │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 32, 32, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 16, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 16, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 16, 16, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 16, 16, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 16, 16, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 16, 16, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 8, 8, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 8, 8, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 8, 8, 128)      │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 8, 8, 128)      │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 8, 8, 128)      │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 8, 8, 128)      │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 4, 4, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 4, 4, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       524,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼─────────────

 Total params: 816,938 (3.12 MB)

 Trainable params: 815,530 (3.11 MB)

 Non-trainable params: 1,408 (5.50 KB)

Epoch 1/40
704/704 ━━━━━━━━━━━━━━━━━━━━ 0s 826ms/step - accuracy: 0.3158 - loss: 2.1541
Epoch 1: val_accuracy improved from None to 0.51800, saving model to best_cifar10_model.keras

Epoch 1: finished saving model to best_cifar10_model.keras
704/704 ━━━━━━━━━━━━━━━━━━━━ 601s 846ms/step - accuracy: 0.3908 - loss: 1.7901 - val_accuracy: 0.5180 - val_loss: 1.3286 - learning_rate: 0.0010
Epoch 2/40
704/704 ━━━━━━━━━━━━━━━━━━━━ 0s 817ms/step - accuracy: 0.5244 - loss: 1.3256
Epoch 2: val_accuracy improved from 0.51800 to 0.60160, saving model to best_cifar10_model.keras

Epoch 2: finished saving model to best_cifar10_model.keras
704/704 ━━━━━━━━━━━━━━━━━━━━ 596s 846ms/step - accuracy: 0.5461 - loss: 1.2668 - val_accuracy: 0.6016 - val_loss: 1.0873 - learning_rate: 0.0010
Epoch 3/40
704/704 ━━━━━━━━━━━━━━━━━━━━ 0s 848ms/step - accuracy: 0.6002 - loss: 1.1325
Epoch 3: val_accuracy improved from 0.60160 to 0.63200, saving model to best_cifar10_model.keras

Epoch 3: finished saving model to bes

In [ ]:
#